# Notebook 4: Drug Repurposing & Virtual Screening
In the final phase of this project, we transition from model development to real-world discovery. The objective is to screen a curated library of human-safe, FDA-approved drugs sourced from the ZINC20 database to identify existing therapeutics that could be repurposed as Androgen Receptor modulators.

By translating these known molecules into 2048-bit Morgan Fingerprints and deploying our champion Random Forest model (trained in Notebook 3), we bypass simple binary classification. Instead, we extract exact binding probabilities, allowing us to mathematically rank the FDA library and confidently isolate the highest-potential candidates for downstream clinical testing.

In [ ]:
import math
from pathlib import Path
from zipfile import ZipFile
from tempfile import TemporaryDirectory

import joblib
import numpy as np
import pandas as pd
import pubchempy as pcp
import time
from rdkit.Chem import PandasTools
from chembl_webresource_client.new_client import new_client
from tqdm.auto import tqdm
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, PandasTools, rdFingerprintGenerator
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

In [ ]:
HERE = Path.cwd()
DATA = HERE / "data"

### 1. Data Cleaning and Pre-processing

To prepare the FDA-Approved drug library for virtual screening, the raw dataset was processed through a rigorous three-step filtration pipeline:

1. **Structural Curation:** The dataset was cleaned by removing duplicate entries and any records lacking valid SMILES strings to ensure computational readability by RDKit.
2. **Physicochemical Filtering (Lipinski’s Rule of Five):** The four core physical descriptors (Molecular Weight, LogP, H-Bond Donors, H-Bond Acceptors) were calculated for each drug. Molecules violating the Rule of Five were discarded to ensure the final candidates are physically viable for oral administration.
3. **PAINS Elimination:** The surviving molecules were screened against the Pan Assay Interference Compounds (PAINS) catalog to prevent false positives in downstream *in vitro* laboratory testing. Standard "unwanted substructure" (toxicophore) filters were intentionally bypassed, as the established clinical safety profiles of FDA-approved drugs override theoretical toxicity alerts.

In [ ]:
# Load the list of FDA-Approved Drugs downloaded from ZINC20
drugs_df = pd.read_csv(DATA / "dbfda-substances.csv")
drugs_df.head()

In [ ]:
# Look at the number of drugs in the list
print(f"DataFrame shape: {drugs_df.shape}")

In [ ]:
# Remove entries with missing SMILES data
drugs_df.dropna(axis=0, how="any", inplace=True)
print(f"DataFrame shape: {drugs_df.shape}")

In [ ]:
# Remove duplicate molecules according to ZINC_ID
drugs_df.drop_duplicates("ZINC_ID", keep="first", inplace=True)
print(f"DataFrame shape: {drugs_df.shape}")

In [ ]:
# Define a function to calculate the Lipinski's Ro5 descriptors
def calculate_ro5_properties(smiles):

    # RDKit molecule from SMILES
    molecule = Chem.MolFromSmiles(smiles)

    # Calculate Ro5-relevant chemical properties
    molecular_weight = Descriptors.ExactMolWt(molecule)
    n_hba = Descriptors.NumHAcceptors(molecule)
    n_hbd = Descriptors.NumHDonors(molecule)
    logp = Descriptors.MolLogP(molecule)

    # Check if Ro5 conditions fulfilled
    conditions = [molecular_weight <= 500, n_hba <= 10, n_hbd <= 5, logp <= 5]
    ro5_fulfilled = sum(conditions) >= 3

    # Return True if no more than one out of four conditions is violated
    return pd.Series(
        [molecular_weight, n_hba, n_hbd, logp, ro5_fulfilled],
        index=["Molecular_weight", "N_hba", "N_hbd", "Logp", "Ro5_fulfilled"],
    )

In [ ]:
# Apply the function to the FDA-Approved drugs dataset
ro5_properties = drugs_df["SMILES"].apply(calculate_ro5_properties)
ro5_properties.head()

In [ ]:
# Merge the Lipinski's Ro5 results into the FDA-Approved drugs dataset
drugs_df = pd.concat([drugs_df, ro5_properties], axis=1)
drugs_df.head()

In [ ]:
# Filter out the drugs that do not fulfill the Lipinski's Rule of Five
drugs_ro5_fulfilled = drugs_df[drugs_df["Ro5_fulfilled"]]
print(f"# Compounds in filtered data set: {drugs_ro5_fulfilled.shape[0]}")

In [ ]:
# Add molecule column to screen for PAINS
PandasTools.AddMoleculeColumnToFrame(drugs_ro5_fulfilled, smilesCol="SMILES")

In [ ]:
# Initialize a blank parameters object to configure our structural
params = FilterCatalogParams()

# Add the official PAINS (Pan Assay Interference Compounds) catalog to our parameters.
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)

# Build the active FilterCatalog engine using our PAINS parameters.
# We will use this 'catalog' object to scan our molecules and flag any rule-breakers.
catalog = FilterCatalog(params)

In [ ]:
# Initialize empty lists to separate the problematic molecules from the safe ones
matches_PAIN = []
clean_PAIN = []

# Iterate through the dataset using tqdm to display a progress bar
for index, row in tqdm(drugs_ro5_fulfilled.iterrows(), total=drugs_ro5_fulfilled.shape[0]):

    # Convert the 1D SMILES string into a 2D RDKit molecule object for structural scanning
    molecule = Chem.MolFromSmiles(row.SMILES)

    # Screen the molecule against the PAINS catalog
    entry = catalog.GetFirstMatch(molecule)  # Get the first matching PAINS
    if entry is not None:
        # If a PAINS substructure is found, record its details
        matches_PAIN.append(
            {
                "zinc_id": row.ZINC_ID,
                "rdkit_molecule": molecule,
                "pains": entry.GetDescription().capitalize(),
            }
        )
    else:
        # If no PAINS are found, save the index of this molecule
        clean_PAIN.append(index)

# Convert the flagged PAINS into their own dataframe
matches_PAIN = pd.DataFrame(matches_PAIN)

# Overwrite our working dataframe to strictly include only the safe, PAINS-free molecules
drugs_df_clean = drugs_ro5_fulfilled.loc[clean_PAIN]

In [ ]:
# Look at the number of drugs left after filtering
print(f"DataFrame shape: {drugs_df_clean.shape}")

# Reset indices of the data frame
drugs_df_clean.reset_index(drop=True, inplace=True)


In [ ]:
drugs_df_clean.head()

### 2. Virtual Screening Deployment

With a pristine dataset of structurally viable, human-safe molecules, we deployed the champion Machine Learning pipeline developed in Notebook 3:

1. **Structural Feature Extraction (Morgan Fingerprints):** The SMILES strings of the surviving drugs were translated into 2048-bit Morgan Fingerprints (Radius 3). This specific structural representation was selected based on its superior cross-validation performance compared to MACCS Keys during the model evaluation phase.
2. **Feature Integration:** The 2048 structural bits were mathematically concatenated with the 4 continuous Lipinski descriptors (Molecular Weight, LogP, H-Bond Acceptors, H-Bond Donors). This created the exact 2052-dimensional feature array our model requires to combine spatial geometry with physical bulk.
3. **Probabilistic Scoring:** The pre-trained Random Forest Classifier was unleashed on the screening library. Rather than relying on simple binary classification, we utilized the model's probabilistic outputs (`predict_proba`) to assign a precise confidence score to each drug. This allowed us to rank the FDA-approved library from highest to lowest probability of Androgen Receptor activation.

In [ ]:
# Create an independent copy of the data to keep our original dataframe pristine
ml_df = drugs_df_clean.copy()

In [ ]:
# Define a function to convert SMILES strings into Morgan fingerprint array for machine learning
def smiles_to_morgan(smiles, n_bits=2048):

    # Convert smiles to RDKit mol object
    mol = Chem.MolFromSmiles(smiles)

    # Initialize the generator for Morgan fingerprints
    fpg = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=n_bits)

    return np.array(fpg.GetFingerprint(mol))

In [ ]:
# Convert the SMILES strings into Morgan fingerprints
ml_df['Morgan_fp'] = ml_df['SMILES'].apply(smiles_to_morgan)

In [ ]:
# Define the Lipinski's Rule of 5 properties
lipinski_cols = ["Molecular_weight", "N_hba", "N_hbd", "Logp"]

# Define a function to combine the Lipinski features with the Morgan fingerprints
def combine_features(row, fp_col):

    # Extract Lipinski features as a float array
    lipinski = row[lipinski_cols].values.astype(float)

    # Extract the fingerprint array
    fp = np.array(row[fp_col])

    # Stitch them together
    return np.concatenate((lipinski, fp))

# Apply the combination function to create our final ML-ready columns
ml_df['Morgan_combined'] = ml_df.apply(lambda row: combine_features(row, 'Morgan_fp'), axis=1)

print(f"Morgan Feature Vector Length: {len(ml_df['Morgan_combined'].iloc[0])}")

In [ ]:
ml_df.head()

In [ ]:
# Extract the combined features for virtual screening
screen_list = list(ml_df['Morgan_combined'])

# Load the Random Forest model trained in Notebook 3
ml_model = joblib.load('morgan_model.pkl')

In [ ]:
# Run the model on the dataset and create a column to save the prediction score
ml_df['Probability_active'] = ml_model.predict_proba(screen_list)[:, 1]

# Sort the dataset by the prediction score
top_hits = ml_df.sort_values(by="Probability_active", ascending=False).reset_index(drop=True)

# Look at the Top 10 active drugs
top_hits[["ZINC_ID","SMILES","Molecular_weight","N_hba","N_hbd","Logp","Probability_active"]].head(10)

In [ ]:
# Search for drug names in NIH PubChem
print("Connecting to PubChem to translate SMILES into English drug names...")

# Define a function that queries the NIH PubChem database to find the common name of a SMILES string.
def fetch_drug_name(smiles):

    try:
        # Search the database using the SMILES string
        compounds = pcp.get_compounds(smiles, namespace='smiles')

        if compounds:
            # Grab the list of known names for this molecule
            synonyms = compounds[0].synonyms
            if synonyms:
                # Return the very first synonym (usually the most common commercial name)
                # We also capitalize it so it looks professional in our table
                return synonyms[0].capitalize()
        return "Name not found"

    except Exception as e:
        return "API Error"

# Take just the top 10 rows to save time and API limits
final_top_10 = top_hits.head(10).copy()

# Apply the translation function
# We add a 1-second pause (sleep) between each so we don't overwhelm the NIH server
names = []
for smi in final_top_10['SMILES']:
    names.append(fetch_drug_name(smi))
    time.sleep(1) # Be polite to the API!

final_top_10['Drug_Name'] = names

# Print the final table!
print("\n🏆 TRANSLATION COMPLETE! 🏆")
final_top_10[['Drug_Name', 'ZINC_ID', 'Probability_active', 'SMILES']]